# casmi26 FPNet full-train rebuild, T4 session 2 (Issue #82)
Fold-safe kit `train_fpnet.py` + session-boundary save/resume only (`train_fpnet_t4.py`, derived and sha-checked in the kernel), `--holdout '' --hours 1000 --max_steps 82855`; other arguments at the kit defaults. CUDA self-check and a wiring rehearsal run before training. Not for v3 evaluation (trains on v3 answers).


In [ ]:
SESSION = 2
PROD_STEPS = 82855
SESSION_HOURS = 10.5
REHEARSAL_HOURS = 0.03
REHEARSAL_MIN_STEPS = 20
SELFCHECK_STEPS = 50
SELFCHECK_NBITS = 10226
FIXED_ARGS = ['--holdout', '', '--hours', '1000', '--max_steps', '82855']
EXPECT = {('dmitriigluzdov/casmi26-fold-safe-fpnet', 'train_fpnet.py'): '46b4dceebb2badeac78e0d5e55849169f5bef8da70a3adf1a7431fe34e603312', ('dmitriigluzdov/casmi26-fold-safe-fpnet', 'split.parquet'): 'dd62549f81b9540890e0da17a06d0dcd216a2781b2c31687bd333445fe57d45e', ('dmitriigluzdov/casmi26-fold-safe-fpnet', 'fp_bits.npy'): 'c3869e86449d6fed6e79e9655f8a7d35bd55c3138ecddfc1dae5072627e78677', ('ahmedberatozer/casmi26-v4b-models', 'code/casmi/fpnet.py'): '0ea6e372d1f045a877d350234c2ccf5d93fdd98cb23bd4c22376bf7e99547608', ('ahmedberatozer/casmi26-v4b-models', 'code/casmi/traindata.py'): 'cecb329d96362f11d16b40a26f1888f092610ff6202812408c13e2c293c6a2bc', ('ahmedberatozer/casmi26-v4b-models', 'code/casmi/spectra.py'): 'e2481761cdfb2515e8a945751f85f6b9e885680a8e974d7ac35c78bbf279ae74', ('ahmedberatozer/casmi26-v4b-models', 'code/casmi/chem.py'): '99474f23d8a67040b0c0f8c70e73592a6f7f656f50103e350ca473451e2169f1', ('ahmedberatozer/casmi26-v2-pool', 'fp_bits.npy'): 'c3869e86449d6fed6e79e9655f8a7d35bd55c3138ecddfc1dae5072627e78677', ('ahmedberatozer/casmi26-v2-pool', 'train_structs.parquet'): 'f20aa1cac23af42e430a6633707fda34e48dea97114bdc3e443897f692df3656', ('ahmedberatozer/casmi26-spec-cache', 'cache/spec_mz.npy'): '482332fec3fabcc46e06ce1b23d79e633af7009b8190a3dcd0a675121b42342b', ('ahmedberatozer/casmi26-spec-cache', 'cache/spec_it.npy'): '9e127f00eb9d4217403991304e69166adc9a36a5e624ce7fd1ddbf6b1417cc23', ('ahmedberatozer/casmi26-spec-cache', 'cache/spec_off.npy'): 'af2be108a532a278ec4563c023808b7fa19e40fdf4ea7dc888e980bb6f618e6c', ('ahmedberatozer/casmi26-spec-cache', 'cache/spec_meta.parquet'): '4f968592dc75b9bf2b6256dda394be42bad3cd07fd243ff0078101e066a44376'}
RECORD_ONLY = [('ahmedberatozer/casmi26-v2-pool', 'pool_meta.parquet'), ('ahmedberatozer/casmi26-v2-pool', 'pool_fp.npy')]
WORK_LINKS = {'spec_off.npy': ('ahmedberatozer/casmi26-spec-cache', 'cache/spec_off.npy'), 'spec_mz.npy': ('ahmedberatozer/casmi26-spec-cache', 'cache/spec_mz.npy'), 'spec_it.npy': ('ahmedberatozer/casmi26-spec-cache', 'cache/spec_it.npy'), 'spec_meta.parquet': ('ahmedberatozer/casmi26-spec-cache', 'cache/spec_meta.parquet'), 'split.parquet': ('dmitriigluzdov/casmi26-fold-safe-fpnet', 'split.parquet'), 'train_structs.parquet': ('ahmedberatozer/casmi26-v2-pool', 'train_structs.parquet'), 'pool_meta.parquet': ('ahmedberatozer/casmi26-v2-pool', 'pool_meta.parquet'), 'pool_fp.npy': ('ahmedberatozer/casmi26-v2-pool', 'pool_fp.npy'), 'fp_bits.npy': ('ahmedberatozer/casmi26-v2-pool', 'fp_bits.npy')}
T4_SHA = '95507246a8be9a3e80ada817dfcdb6d8d1b110461a304436f83a134399e2f330'
FR_SHA = '179666c097cf5d08ce4c15967aa2d055884e4fef6918ae890578a5cf3ce2038e'
REPORT_SRC = '"""Issue #82: キットの train_fpnet.py のログから it/s と時間を読み、本番 82,855 ステップの時間を外挿する。\n\nノートブック（kernels/fpnet_full_train）にこのファイルの中身を埋め込んで使う。外挿の式は Issue #82 の事前登録のとおり:\n    本番の時間 = 準備とデータの読み込みの時間 + 82,855 / 定常の it/s + 16 x 最後の評価にかかった時間\n定常の it/s = ログの step 1000 の行から最後の step の行までのステップの差 / 時刻の差。\n"""\nimport json\nimport re\n\nPROD_STEPS = 82855\nN_EVALS = 16\nSESSION_LIMIT_H = 11.0\nSTEADY_FROM = 1000\n\n_TS = r\'(\\d\\d):(\\d\\d):(\\d\\d)\'\n_STEP = re.compile(_TS + r\' step (\\d+) lr \\S+ .* ([0-9.]+) it/s queue (\\d+)\\s*$\')\n_DATA = re.compile(r\'\\[Data\\] train spectra ([\\d,]+) groups ([\\d,]+) pool ([\\d,]+) nbits (\\d+)\')\n_SETS = re.compile(_TS + r" (\\{.*\\}) train spectra (\\d+)\\s*$")\n_PARAMS = re.compile(_TS + r\' parameters (\\d+) \')\n_EVAL = re.compile(_TS + r\' EVAL (\\{.*\\})\\s*$\')\n\n\ndef _sec(h, m, s):\n    return int(h) * 3600 + int(m) * 60 + int(s)\n\n\ndef _unwrap(times):\n    """HH:MM:SS の秒を、日付をまたいでも増えるように直す。"""\n    out, add, prev = [], 0, None\n    for t in times:\n        if prev is not None and t + add < prev:\n            add += 86400\n        out.append(t + add)\n        prev = t + add\n    return out\n\n\ndef parse_log(text):\n    steps, params_t, eval_t, eval_rec, data, sets = [], None, None, None, None, None\n    for line in text.splitlines():\n        m = _STEP.match(line)\n        if m:\n            steps.append((_sec(*m.group(1, 2, 3)), int(m.group(4)), float(m.group(5)), int(m.group(6))))\n            continue\n        m = _DATA.search(line)\n        if m:\n            data = dict(train_spectra=int(m.group(1).replace(\',\', \'\')), groups=int(m.group(2).replace(\',\', \'\')),\n                        pool=int(m.group(3).replace(\',\', \'\')), nbits=int(m.group(4)))\n            continue\n        m = _SETS.match(line)\n        if m:\n            sets = dict(eval_sets=m.group(4), train_spectra=int(m.group(5)), t=_sec(*m.group(1, 2, 3)))\n            continue\n        m = _PARAMS.match(line)\n        if m:\n            params_t = _sec(*m.group(1, 2, 3))\n            continue\n        m = _EVAL.match(line)\n        if m:\n            eval_t = _sec(*m.group(1, 2, 3))\n            eval_rec = json.loads(m.group(4))\n    return dict(steps=steps, params_t=params_t, eval_t=eval_t, eval=eval_rec, data=data, sets=sets)\n\n\ndef steady_rate(steps, start=STEADY_FROM):\n    """step >= start の最初の行から最後の行までの (ステップの差 / 時刻の差)。行が足りなければ None。"""\n    if not steps:\n        return None\n    ts = _unwrap([s[0] for s in steps])\n    idx = [i for i, s in enumerate(steps) if s[1] >= start]\n    if len(idx) < 2:\n        return None\n    a, b = idx[0], len(steps) - 1\n    dt = ts[b] - ts[a]\n    return (steps[b][1] - steps[a][1]) / dt if dt > 0 else None\n\n\ndef extrapolate(setup_seconds, rate, eval_seconds, prod_steps=PROD_STEPS, n_evals=N_EVALS):\n    """事前登録の式。rate が無ければ None。"""\n    if not rate:\n        return None\n    total = setup_seconds + prod_steps / rate + n_evals * eval_seconds\n    return dict(setup_seconds=setup_seconds, steady_it_per_s=rate, eval_seconds=eval_seconds, prod_steps=prod_steps,\n                n_evals=n_evals, train_seconds=prod_steps / rate, total_seconds=total, total_hours=total / 3600.0,\n                one_session=bool(total / 3600.0 <= SESSION_LIMIT_H), session_limit_hours=SESSION_LIMIT_H)\n\n\ndef summarize(text, proc_start_sec_of_day):\n    """ログ全体から報告用の数をまとめる。proc_start_sec_of_day はログと同じ時計の、学習プロセスを起動した時刻（その日の秒）。"""\n    p = parse_log(text)\n    steps = p[\'steps\']\n    rate = steady_rate(steps)\n    setup = None\n    if p[\'params_t\'] is not None:\n        setup = p[\'params_t\'] - proc_start_sec_of_day\n        if setup < 0:\n            setup += 86400\n    eval_s, lower_bound = 0.0, True\n    if p[\'eval_t\'] is not None and steps:\n        eval_s = p[\'eval_t\'] - steps[-1][0]\n        if eval_s < 0:\n            eval_s += 86400\n        lower_bound = False\n    ext = extrapolate(setup or 0.0, rate, eval_s)\n    if ext is not None:\n        ext[\'lower_bound\'] = lower_bound or setup is None\n    return dict(last_step=steps[-1][1] if steps else 0, last_logged_it_per_s=steps[-1][2] if steps else None,\n                last_queue=steps[-1][3] if steps else None, min_queue=min(s[3] for s in steps) if steps else None,\n                data=p[\'data\'], sets=p[\'sets\'], final_eval=p[\'eval\'], extrapolation=ext)\n'
MAKE_T4_SRC = '"""Derive train_fpnet_t4.py from the kit\'s train_fpnet.py by exact, asserted replacements (Issue #82, 11/21).\n\n    python scripts/train/fpnet_kit/make_t4.py <kit train_fpnet.py> <out train_fpnet_t4.py>\n\nAdds only session-boundary save/resume (every added line ends with \'# [t4:resume]\'); device, precision, batch, losses,\nhyper-parameters and the schedule stay as in the kit. The kernel runs this file on the dataset\'s original (sha256 checked)\nand requires the result to equal the committed train_fpnet_t4.py.\n"""\nimport sys\nfrom pathlib import Path\n\nTAG = \'  # [t4:resume]\'\n\n\ndef derive(src: str) -> str:\n    src = src.replace(\'\\r\\n\', \'\\n\')\n\n    def rep(old, new_lines):\n        nonlocal src\n        assert src.count(old) == 1, (src.count(old), old)\n        src = src.replace(old, \'\\n\'.join(new_lines))\n\n    def m(lines):\n        return [l + TAG if l else l for l in lines]\n\n    # helpers module next to this file (fpnet_resume.py)\n    rep(\'import numpy as np\\n\',\n        [\'import numpy as np\'] +\n        m(["sys.path.insert(0, os.path.dirname(os.path.abspath(__file__))); import fpnet_resume as FR"]) + [\'\'])\n    # new arguments (session number, session length, resume file); kit hyper-parameters untouched\n    rep("ap.add_argument(\'--bias_init\', type=int, default=0)\\n",\n        ["ap.add_argument(\'--bias_init\', type=int, default=0)"] +\n        m(["ap.add_argument(\'--session\', type=int, default=1); ap.add_argument(\'--session_hours\', type=float, default=10.5)",\n           "ap.add_argument(\'--resume_from\', default=\'\')"]) + [\'\'])\n    # worker RNG: kit formula in a fresh run, seed*1000+wid+1 + 7919*session after a resume (pre-registered on #82)\n    rep("    D.rng = np.random.default_rng(args.seed * 1000 + wid + 1)\\n",\n        m(["    D.rng = np.random.default_rng(FR.worker_seed(args.seed, wid, args.session, G[\'resumed\']))"]) + [\'\'])\n    # read the resume file before the workers fork (they need to know whether this is a resumed session)\n    rep("    G[\'D\'] = D\\n",\n        ["    G[\'D\'] = D"] +\n        m(["    ck = FR.load_resume(args.resume_from) if args.resume_from else None",\n           "    assert (ck is not None) == (args.session > 1), (args.session, args.resume_from)",\n           "    if ck is not None:",\n           "        same = (\'session\', \'session_hours\', \'resume_from\', \'code\', \'run\')",\n           "        bad = {k: (ck[\'args\'].get(k), v) for k, v in vars(args).items() if k not in same and ck[\'args\'].get(k) != v}",\n           "        assert not bad and ck[\'session\'] == args.session - 1, (bad, ck[\'session\'], args.session)",\n           "    G[\'resumed\'] = ck is not None"]) + [\'\'])\n    # restore model / AdamW / GradScaler / step / elapsed / history / EMA / RNG once they exist\n    rep("    ema = {k: 0.0 for k in (\'loss\', \'rank\', \'bce\', \'el\', \'acc\')}\\n",\n        ["    ema = {k: 0.0 for k in (\'loss\', \'rank\', \'bce\', \'el\', \'acc\')}"] +\n        m(["    lr_check = None",\n           "    if ck is not None:",\n           "        step, elapsed, hist, ema, lr_check = FR.apply_resume(ck, model, opt, scaler)",\n           "        t0 = time.time() - elapsed",\n           "        log(\'resumed from\', args.resume_from, \'session\', args.session, \'step\', step, \'elapsed_s\', round(elapsed, 1))",\n           "        del ck",\n           "    t_sess = time.time()"]) + [\'\'])\n    # session boundary: after this step\'s lr is set, before the batch is taken\n    rep("        for g in opt.param_groups:\\n            g[\'lr\'] = lr\\n        arrs, pos, negs = q.get()\\n",\n        ["        for g in opt.param_groups:", "            g[\'lr\'] = lr"] +\n        m(["        if lr_check is not None:",\n           "            assert lr == lr_check, (lr, lr_check)",\n           "            log(\'lr continues\', f\'{lr:.10e}\', \'step\', step); lr_check = None",\n           "        if time.time() - t_sess >= args.session_hours * 3600:",\n           "            assert all(math.isfinite(v) for v in ema.values()), ema",\n           "            FR.save_resume(f\'{args.tag}_resume.pt\', model, opt, scaler, step, time.time() - t0, hist, ema, lr, args.session, vars(args))",\n           "            json.dump(hist, open(f\'{args.tag}_history.json\', \'w\'), indent=1)",\n           "            log(\'SESSION_DONE\', \'session\', args.session, \'step\', step, \'lr\', f\'{lr:.10e}\', \'elapsed_s\', round(time.time() - t0, 1))",\n           "            for p in procs:",\n           "                p.terminate()",\n           "            return"]) +\n        ["        arrs, pos, negs = q.get()", \'\'])\n    return src\n\n\nif __name__ == \'__main__\':\n    out = derive(Path(sys.argv[1]).read_bytes().decode())\n    Path(sys.argv[2]).write_text(out)\n    print(\'ok\')\n'
FR_SRC = '"""Session-boundary save/resume helpers for the kit\'s train_fpnet.py on Kaggle T4 (Issue #82, 11/21 pre-registration).\n\nUsed by train_fpnet_t4.py (derived from the kit file by make_t4.py) and by the CUDA self-check cell of the kernel.\n\n- save_resume / load_resume / apply_resume: model, AdamW and torch.amp.GradScaler state_dicts, step, training elapsed\n  seconds, eval history, loss EMA, lr, session, and the torch (CPU, CUDA) / numpy / python RNG states.\n  Written to <path>.tmp and moved with os.replace. The kit has no lr-scheduler object: lr is a pure function of step\n  and elapsed time (prog = max(step / max_steps, elapsed / budget)), so restoring step and elapsed continues it.\n- worker_seed: the kit\'s worker seed (seed*1000+wid+1) in a fresh run; seed*1000+wid+1 + 7919*session after a resume.\n- state_equal: bitwise comparison of nested state (tensors by dtype, shape and bytes).\n- selfcheck: train a separate model copy for N steps, save, load into fresh objects, compare bitwise, run one more\n  step; the caller\'s RNG state is restored on exit.\n"""\nfrom __future__ import annotations\n\nimport math\nimport os\nimport random\nfrom contextlib import contextmanager\n\nimport numpy as np\nimport torch\n\nRESUME_PRIME = 7919\nFORMAT = \'fpnet_t4_resume_v1\'\n\n\ndef worker_seed(seed: int, wid: int, session: int, resumed: bool) -> int:\n    if not resumed:\n        return seed * 1000 + wid + 1                       # kit formula, unchanged\n    return seed * 1000 + wid + 1 + RESUME_PRIME * session  # pre-registered on #82 (11/21)\n\n\ndef capture_rng() -> dict:\n    st = dict(torch_cpu=torch.get_rng_state(), numpy=np.random.get_state(), python=random.getstate(), cuda=None)\n    if torch.cuda.is_available() and torch.cuda.is_initialized():\n        st[\'cuda\'] = torch.cuda.get_rng_state_all()\n    return st\n\n\ndef restore_rng(st: dict) -> None:\n    torch.set_rng_state(st[\'torch_cpu\'])\n    np.random.set_state(st[\'numpy\'])\n    random.setstate(st[\'python\'])\n    if st.get(\'cuda\') is not None and torch.cuda.is_available():\n        torch.cuda.set_rng_state_all(st[\'cuda\'])\n\n\n@contextmanager\ndef preserved_rng():\n    st = capture_rng()\n    try:\n        yield\n    finally:\n        restore_rng(st)\n\n\ndef save_resume(path, model, opt, scaler, step, elapsed, hist, ema, lr, session, args=None) -> dict:\n    d = dict(format=FORMAT, model=model.state_dict(), opt=opt.state_dict(), scaler=scaler.state_dict(), step=int(step),\n             elapsed=float(elapsed), hist=list(hist), ema=dict(ema), lr=float(lr), session=int(session),\n             rng=capture_rng(), args=dict(args) if args is not None else None)\n    tmp = str(path) + \'.tmp\'\n    torch.save(d, tmp)\n    os.replace(tmp, path)\n    return d\n\n\ndef load_resume(path) -> dict:\n    ck = torch.load(path, map_location=\'cpu\', weights_only=False)\n    assert ck.get(\'format\') == FORMAT, ck.get(\'format\')\n    return ck\n\n\ndef apply_resume(ck, model, opt, scaler, restore_rng_state=True):\n    """Load the state into existing objects; returns (step, elapsed, hist, ema, lr)."""\n    model.load_state_dict(ck[\'model\'])\n    opt.load_state_dict(ck[\'opt\'])\n    scaler.load_state_dict(ck[\'scaler\'])\n    if restore_rng_state:\n        restore_rng(ck[\'rng\'])\n    return ck[\'step\'], ck[\'elapsed\'], list(ck[\'hist\']), dict(ck[\'ema\']), ck[\'lr\']\n\n\ndef state_equal(a, b, path=\'\') -> list:\n    """List of paths where a and b differ (empty = bitwise equal)."""\n    if isinstance(a, torch.Tensor) or isinstance(b, torch.Tensor):\n        if not (isinstance(a, torch.Tensor) and isinstance(b, torch.Tensor)):\n            return [path + \': type\']\n        a = a.detach().cpu(); b = b.detach().cpu()\n        if a.dtype != b.dtype or a.shape != b.shape:\n            return [path + f\': {a.dtype}{tuple(a.shape)} vs {b.dtype}{tuple(b.shape)}\']\n        return [] if a.contiguous().reshape(-1).view(torch.uint8).equal(b.contiguous().reshape(-1).view(torch.uint8)) else [path + \': bytes\']\n    if isinstance(a, np.ndarray) or isinstance(b, np.ndarray):\n        ok = isinstance(a, np.ndarray) and isinstance(b, np.ndarray) and a.dtype == b.dtype and a.shape == b.shape \\\n            and a.tobytes() == b.tobytes()\n        return [] if ok else [path + \': ndarray\']\n    if isinstance(a, dict) and isinstance(b, dict):\n        if set(a) != set(b):\n            return [path + f\': keys {sorted(map(str, set(a) ^ set(b)))}\']\n        out = []\n        for k in a:\n            out += state_equal(a[k], b[k], f\'{path}/{k}\')\n        return out\n    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):\n        if len(a) != len(b):\n            return [path + \': len\']\n        out = []\n        for i, (x, y) in enumerate(zip(a, b)):\n            out += state_equal(x, y, f\'{path}[{i}]\')\n        return out\n    if isinstance(a, float) and isinstance(b, float) and math.isnan(a) and math.isnan(b):\n        return []\n    return [] if (type(a) == type(b) and a == b) else [path + f\': {a!r} vs {b!r}\']\n\n\ndef kit_lr(step, max_steps, elapsed, budget, lr0, warmup):\n    """The kit\'s inline lr formula (train_fpnet.py, main loop), for tests and the self-check."""\n    prog = max(step / max_steps, elapsed / budget)\n    return lr0 * min(1.0, (step + 1) / warmup) * (0.02 + 0.98 * 0.5 * (1 + math.cos(math.pi * prog)))\n\n\ndef make_opt(model, lr=3e-4, wd=0.01):\n    """AdamW exactly as the kit builds it (decay on ndim >= 2, betas 0.9/0.98)."""\n    decay = [p for n, p in model.named_parameters() if p.ndim >= 2]\n    no_decay = [p for n, p in model.named_parameters() if p.ndim < 2]\n    return torch.optim.AdamW([dict(params=decay, weight_decay=wd), dict(params=no_decay, weight_decay=0.0)],\n                             lr=lr, betas=(0.9, 0.98))\n\n\ndef train_step(model, opt, scaler, batch, loss_fn, lr, device, dtype):\n    """One optimizer step with the kit\'s order: set lr, autocast forward, scaled backward, unscale, clip 1.0, step."""\n    for g in opt.param_groups:\n        g[\'lr\'] = lr\n    with torch.autocast(device, dtype=dtype, enabled=dtype != torch.float32):\n        out = model(*batch[\'inp\'])\n    loss = loss_fn(out, batch)\n    opt.zero_grad(set_to_none=True)\n    scaler.scale(loss).backward()\n    scaler.unscale_(opt)\n    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n    scaler.step(opt)\n    scaler.update()\n    return loss.item()\n\n\ndef selfcheck(make_model, make_batch, loss_fn, device, dtype, path, steps=50, seed=20261121, lr0=3e-4, warmup=2000,\n              max_steps=82855):\n    """Separate model copy and RNG state; N steps -> save -> load into fresh objects -> bitwise compare -> 1 more step.\n\n    make_batch(step, generator) must build the batch from the given torch.Generator only. The caller\'s RNG state\n    (torch CPU/CUDA, numpy, python) is the same before and after the call.\n    """\n    with preserved_rng():\n        torch.manual_seed(seed)\n        gen = torch.Generator(device=\'cpu\').manual_seed(seed + 1)\n        model = make_model().to(device)\n        opt = make_opt(model, lr=lr0)\n        scaler = torch.amp.GradScaler(device, enabled=True)\n        losses = []\n        for step in range(steps):\n            lr = kit_lr(step, max_steps, 0.0, 3.6e6, lr0, warmup)\n            losses.append(train_step(model, opt, scaler, make_batch(step, gen), loss_fn, lr, device, dtype))\n        lr_next = kit_lr(steps, max_steps, 0.0, 3.6e6, lr0, warmup)\n        saved = save_resume(path, model, opt, scaler, steps, 0.0, [], dict(loss=losses[-1]), lr_next, 1)\n        ck = load_resume(path)\n        model2 = make_model().to(device)\n        opt2 = make_opt(model2, lr=lr0)\n        scaler2 = torch.amp.GradScaler(device, enabled=True)\n        step2, _, _, _, lr2 = apply_resume(ck, model2, opt2, scaler2)\n        diffs = {k: state_equal(a, b) for k, (a, b) in dict(\n            model=(model.state_dict(), model2.state_dict()), opt=(opt.state_dict(), opt2.state_dict()),\n            scaler=(scaler.state_dict(), scaler2.state_dict()), file_vs_live_model=(saved[\'model\'], ck[\'model\'])).items()}\n        nxt = train_step(model2, opt2, scaler2, make_batch(steps, gen), loss_fn, kit_lr(step2, max_steps, 0.0, 3.6e6, lr0, warmup),\n                         device, dtype)\n        res = dict(steps=steps, step_loaded=step2, lr_saved=lr_next, lr_loaded=lr2, scaler_state=scaler2.state_dict(),\n                   diffs=diffs, next_loss=nxt, losses_first_last=[losses[0], losses[-1]],\n                   ok=all(not v for v in diffs.values()) and step2 == steps and lr2 == lr_next and math.isfinite(nxt)\n                   and all(math.isfinite(x) for x in losses))\n        del model, model2, opt, opt2, ck, saved\n        os.remove(path)\n    return res\n'


In [ ]:
# ---- Issue #82: Engine 1 FP bank rebuilt on all train with the fold-safe kit's train_fpnet.py, T4 session SESSION ----
import json, os, sys, time, hashlib, shutil, subprocess, threading, signal, platform
from pathlib import Path
T0 = time.time()
log = lambda *a: print(time.strftime('%H:%M:%S'), *a, flush=True)
OUT = Path('/kaggle/working'); RUN = OUT / 'run'; WORK = RUN / 'work'; CODE = OUT / 'code'; KITD = CODE / 'kit'
REH = Path('/tmp/fpnet_rehearsal')
WORK.mkdir(parents=True, exist_ok=True)
REPORT = dict(issue=82, recipe='kit train_fpnet.py + session-boundary save/resume only (train_fpnet_t4.py)', session=SESSION,
              fixed_args=FIXED_ARGS, session_hours=SESSION_HOURS, status='started')
REPORT_PATH = OUT / f'fpnet_full_report_s{SESSION}.json'
def write_report():
    REPORT['seconds_total_notebook'] = round(time.time() - T0, 1)
    REPORT_PATH.write_text(json.dumps(REPORT, indent=2, default=str))

def ds_root(slug):
    owner, name = slug.split('/')
    for p in (Path('/kaggle/input') / name, Path('/kaggle/input/datasets') / owner / name):
        if p.is_dir():
            return p
    hits = [p for p in Path('/kaggle/input').rglob(name) if p.is_dir()]
    assert len(hits) == 1, (slug, hits)
    return hits[0]

def sha256(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(8 << 20), b''):
            h.update(b)
    return h.hexdigest()

def path_of(slug, rel):
    p = ds_root(slug) / rel
    assert p.is_file(), p
    return p

# 1) inputs: published sha256 must match before training
t = time.time(); hashes = {}
for (slug, rel), want in EXPECT.items():
    got = sha256(path_of(slug, rel)); hashes[f'{slug}/{rel}'] = got
    assert got == want, (slug, rel, got, want)
for slug, rel in RECORD_ONLY:
    hashes[f'{slug}/{rel}'] = sha256(path_of(slug, rel))
REPORT['input_sha256'] = hashes; REPORT['seconds_hash_inputs'] = round(time.time() - t, 1)
log('INPUTS OK', len(EXPECT), 'checked,', len(RECORD_ONLY), 'recorded', REPORT['seconds_hash_inputs'], 's')

# 2) code: copy of v4b code/casmi (writable for numba cache); train_fpnet_t4.py derived from the kit file here
def link_work(work):
    work.mkdir(parents=True, exist_ok=True)
    for name, (slug, rel) in WORK_LINKS.items():
        dst = work / name
        if not dst.exists():
            os.symlink(path_of(slug, rel), dst)
link_work(WORK)
if CODE.exists():
    shutil.rmtree(CODE)
shutil.copytree(ds_root('ahmedberatozer/casmi26-v4b-models') / 'code' / 'casmi', CODE / 'casmi')
for rel in ('fpnet.py', 'traindata.py', 'spectra.py', 'chem.py'):
    assert sha256(CODE / 'casmi' / rel) == EXPECT[('ahmedberatozer/casmi26-v4b-models', 'code/casmi/' + rel)], rel
KITD.mkdir(parents=True)
ns = {'__name__': 'make_t4'}
exec(compile(MAKE_T4_SRC, 'make_t4', 'exec'), ns)
t4_src = ns['derive'](path_of('dmitriigluzdov/casmi26-fold-safe-fpnet', 'train_fpnet.py').read_bytes().decode())
assert hashlib.sha256(t4_src.encode()).hexdigest() == T4_SHA, 'derived train_fpnet_t4.py differs from the committed one'
(KITD / 'train_fpnet_t4.py').write_text(t4_src); (KITD / 'fpnet_resume.py').write_text(FR_SRC)
assert hashlib.sha256((KITD / 'fpnet_resume.py').read_bytes()).hexdigest() == FR_SHA
TRAIN = KITD / 'train_fpnet_t4.py'
REPORT['code_sha256'] = dict(train_fpnet_t4=T4_SHA, fpnet_resume=FR_SHA)

# 3) environment
import torch
REPORT['env'] = dict(python=sys.version.split()[0], torch=torch.__version__, cuda=torch.version.cuda,
                     gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
                     gpu_capability=list(torch.cuda.get_device_capability(0)) if torch.cuda.is_available() else None,
                     gpu_count=torch.cuda.device_count(), vcpu_os_cpu_count=os.cpu_count(),
                     vcpu_affinity=len(os.sched_getaffinity(0)), workers=16, workers_note='kit default, not passed',
                     platform=platform.platform(), torch_config=torch.__config__.show())
try:
    REPORT['env']['mem_total_gb'] = round(int(open('/proc/meminfo').read().split('MemTotal:')[1].split()[0]) / 2**20, 2)
except Exception as e:
    REPORT['env']['mem_total_gb'] = repr(e)
log('ENV', {k: v for k, v in REPORT['env'].items() if k != 'torch_config'})

# 4) CUDA self-check: a separate FPNet copy (same shape, own RNG state) -> 50 steps -> save -> load -> bitwise -> 1 step
sys.path.insert(0, str(CODE)); sys.path.insert(0, str(KITD))
import numpy as np
import torch.nn.functional as F
import fpnet_resume as FR
from casmi import fpnet
dev = 'cuda'; dtype = fpnet.autocast_dtype(dev)
def make_model():
    return fpnet.FPNet(SELFCHECK_NBITS, d=512, layers=8)
def make_batch(step, gen):
    items = []
    for _ in range(8):
        n = 40
        prec = float(150 + 600 * torch.rand(1, generator=gen).item())
        mz = np.sort((50 + (prec - 50) * torch.rand(n, generator=gen)).numpy()).astype(np.float32)
        it = torch.rand(n, generator=gen).numpy().astype(np.float32)
        items.append(dict(mz=mz, it=it, prec=prec, adduct_ix=int(torch.randint(0, fpnet.N_ADDUCT, (1,), generator=gen)),
                          ce=30.0, ce_known=1.0, n_merged=1, mode=1.0))
    y = (torch.rand(8, SELFCHECK_NBITS, generator=gen) > 0.9).float().to(dev)
    return dict(inp=fpnet.collate(items, dev), y=y, e=torch.rand(8, 10, generator=gen).to(dev))
def loss_fn(out, b):
    z, el = out
    return F.binary_cross_entropy_with_logits(z.float(), b['y']) + 0.1 * F.smooth_l1_loss(el.float(), b['e'])
t = time.time()
sc = FR.selfcheck(make_model, make_batch, loss_fn, dev, dtype, '/tmp/selfcheck_resume.pt', steps=SELFCHECK_STEPS)
sc['seconds'] = round(time.time() - t, 1); sc['dtype'] = str(dtype); sc['nbits'] = SELFCHECK_NBITS
(OUT / 'resume_selfcheck.json').write_text(json.dumps(sc, indent=2, default=str))
REPORT['selfcheck'] = dict(ok=sc['ok'], seconds=sc['seconds'], next_loss=sc['next_loss'], dtype=str(dtype))
log('SELFCHECK', json.dumps(REPORT['selfcheck']))
torch.cuda.empty_cache()
if not sc['ok']:
    REPORT['status'] = 'selfcheck_failed'; write_report()
    raise SystemExit('self-check failed: no training')

# 5) memory sampler (every 5 s): GPU via nvidia-smi, CPU = RSS sum of the training process tree and system used
import psutil
def sampler(pid, peak, stop):
    while not stop.is_set():
        try:
            g = subprocess.run(['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader,nounits'],
                               capture_output=True, text=True, timeout=10).stdout.split()
            peak['gpu_mem_used_mib'] = max([peak['gpu_mem_used_mib']] + [int(x) for x in g])
        except Exception:
            pass
        try:
            root = psutil.Process(pid); procs = [root] + root.children(recursive=True); rss = 0
            for p in procs:
                try:
                    rss += p.memory_info().rss
                except psutil.Error:
                    pass
            peak['proc_tree_rss_gb'] = max(peak['proc_tree_rss_gb'], rss / 2**30)
        except psutil.Error:
            pass
        vm = psutil.virtual_memory()
        peak['system_used_gb'] = max(peak['system_used_gb'], (vm.total - vm.available) / 2**30)
        peak['samples'] += 1
        stop.wait(5)

def run_train(run, extra, log_path, timeout_s=None):
    cmd = [sys.executable, '-u', str(TRAIN), '--code', str(CODE), '--run', str(run)] + FIXED_ARGS + extra
    peak = dict(gpu_mem_used_mib=0, proc_tree_rss_gb=0.0, system_used_gb=0.0, samples=0); stop = threading.Event()
    lt = time.localtime(); start_sec = lt.tm_hour * 3600 + lt.tm_min * 60 + lt.tm_sec; t_start = time.time(); killed = False
    with open(log_path, 'w') as lf:
        proc = subprocess.Popen(cmd, stdout=lf, stderr=subprocess.STDOUT, start_new_session=True,
                                env=dict(os.environ, NUMBA_CACHE_DIR=str(OUT / 'numba_cache')))
        th = threading.Thread(target=sampler, args=(proc.pid, peak, stop), daemon=True); th.start()
        while proc.poll() is None:
            time.sleep(5)
            if timeout_s and time.time() - t_start > timeout_s:
                killed = True; os.killpg(proc.pid, signal.SIGKILL); proc.wait()
        try:
            os.killpg(proc.pid, signal.SIGKILL)   # daemon data workers left behind after a normal exit
        except ProcessLookupError:
            pass
    stop.set(); th.join(timeout=20)
    return dict(cmd=cmd, returncode=proc.returncode, killed=killed, seconds=round(time.time() - t_start, 1), start_sec=start_sec,
                peak={k: (round(v, 3) if isinstance(v, float) else v) for k, v in peak.items()}, text=Path(log_path).read_text())

# 6) wiring rehearsal: the derived script itself on real data, --seed 1 --tag rehearsal, outside /kaggle/working, deleted
shutil.rmtree(REH, ignore_errors=True); link_work(REH / 'work')
common = ['--seed', '1', '--tag', 'rehearsal', '--session_hours', str(REHEARSAL_HOURS)]
ra = run_train(REH, common, REH / 'a.log', timeout_s=1200)
ka = FR.load_resume(REH / 'rehearsal_resume.pt')['step'] if (REH / 'rehearsal_resume.pt').exists() else -1
rb = run_train(REH, common + ['--session', '2', '--resume_from', str(REH / 'rehearsal_resume.pt')], REH / 'b.log', timeout_s=1200)
ckb = FR.load_resume(REH / 'rehearsal_resume.pt') if (REH / 'rehearsal_resume.pt').exists() else None
reh = dict(a_rc=ra['returncode'], a_seconds=ra['seconds'], a_step=ka, a_session_done='SESSION_DONE' in ra['text'],
           b_rc=rb['returncode'], b_seconds=rb['seconds'], b_resumed_line=f"step {ka} elapsed_s" in rb['text'],
           b_lr_continues='lr continues' in rb['text'], b_session_done='SESSION_DONE' in rb['text'],
           b_step=ckb['step'] if ckb else -1, b_session=ckb['session'] if ckb else -1,
           a_peak=ra['peak'], b_peak=rb['peak'], a_tail=ra['text'][-1500:], b_tail=rb['text'][-1500:])
reh['ok'] = bool(reh['a_rc'] == 0 and reh['a_session_done'] and ka >= REHEARSAL_MIN_STEPS and reh['b_rc'] == 0
                 and reh['b_resumed_line'] and reh['b_lr_continues'] and reh['b_session_done'] and reh['b_step'] > ka
                 and reh['b_session'] == 2 and not ra['killed'] and not rb['killed'])
del ckb
REPORT['rehearsal'] = reh
shutil.rmtree(REH, ignore_errors=True)
log('REHEARSAL', json.dumps({k: v for k, v in reh.items() if not k.endswith('_tail')}))
if not reh['ok']:
    REPORT['status'] = 'rehearsal_failed'; write_report()
    print(reh['a_tail']); print(reh['b_tail'])
    raise SystemExit('rehearsal failed: no training')

# 7) session 2: resume file from session 1's output (kernel_sources), sha256 checked against session 1's report
extra = ['--session_hours', str(SESSION_HOURS)]
if SESSION >= 2:
    reps = [p for p in Path('/kaggle/input').rglob(f'fpnet_full_report_s{SESSION - 1}.json')]
    assert len(reps) == 1, reps
    prev = json.loads(reps[0].read_text())
    assert prev['status'] == 'session_done' and prev['session'] == SESSION - 1, (prev['status'], prev['session'])
    src_ck = reps[0].parent / 'fpnet_resume.pt'
    got = sha256(src_ck)
    assert got == prev['resume_sha256'], (got, prev['resume_sha256'])
    shutil.copy(src_ck, RUN / 'fpnet_resume.pt')
    REPORT['resumed_from'] = dict(report=str(reps[0]), resume_sha256=got, step=prev['resume_step'])
    extra += ['--session', str(SESSION), '--resume_from', str(RUN / 'fpnet_resume.pt')]
del torch

# 8) train (kit hyper-parameters at their defaults; seed 0 from the kit default in session 1)
LOG = OUT / f'train_fpnet_s{SESSION}.log'
r = run_train(RUN, extra, LOG)
text = r['text']; print(text[-3000:])
REPORT['command'] = r['cmd']; REPORT['returncode'] = r['returncode']; REPORT['seconds_train_process'] = r['seconds']
REPORT['peak'] = r['peak']
ns = {}
exec(compile(REPORT_SRC, 'fpnet_full_report', 'exec'), ns)
REPORT['summary'] = ns['summarize'](text, r['start_sec'])
done, sdone = 'ALL_DONE' in text, 'SESSION_DONE' in text
if r['returncode'] == 0 and done:
    REPORT['status'] = 'ok'
elif r['returncode'] == 0 and sdone and (RUN / 'fpnet_resume.pt').exists():
    REPORT['status'] = 'session_done'
else:
    REPORT['status'] = 'failed'
if (RUN / 'fpnet_history.json').exists():
    shutil.copy(RUN / 'fpnet_history.json', OUT / 'fpnet_history.json')
for name in ('fpnet_final.pt', 'fpnet_last.pt') + (('fpnet_resume.pt',) if REPORT['status'] == 'session_done' else ()):
    if (RUN / name).exists():
        shutil.copy(RUN / name, OUT / name)
        REPORT[name.replace('.pt', '') + '_sha256'] = sha256(OUT / name)
if REPORT['status'] == 'session_done':
    import torch
    ck = torch.load(OUT / 'fpnet_resume.pt', map_location='cpu', weights_only=False)
    REPORT['resume_sha256'] = REPORT.pop('fpnet_resume_sha256'); REPORT['resume_step'] = ck['step']
    REPORT['resume_lr'] = ck['lr']; REPORT['resume_elapsed_s'] = ck['elapsed']; del ck
shutil.rmtree(CODE, ignore_errors=True); shutil.rmtree(OUT / 'numba_cache', ignore_errors=True)
shutil.rmtree(RUN, ignore_errors=True)   # work/ = symlinks only; .pt files were copied to /kaggle/working above
write_report()
log('REPORT', json.dumps({k: v for k, v in REPORT.items() if k not in ('input_sha256', 'env', 'command', 'rehearsal')}, default=str)[:4000])
assert REPORT['status'] in ('ok', 'session_done'), REPORT['status']
